In [5]:
import os
from typing import List,Dict,Any
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

In [9]:
## create a simple document
doc=Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Krish Naik",
        "date_created":"2024-01-01",
        "cutom_field":"any_value"

    }
)
print("Document Structure")

print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

# Why metadata matters:
print("\n Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Content :This is the main text content that will be embedded and searched.
Metadata :{'source': 'example.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01', 'cutom_field': 'any_value'}

 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [10]:
print(type(doc))

<class 'langchain_core.documents.base.Document'>


### Reading Text Files

#### 1.Text Loader

In [11]:
from langchain_community.document_loaders import TextLoader

In [19]:
loader = TextLoader(r'D:\RAG Projects\Build-a-RAG-Chatbot\data\text_files\python_intro.txt',encoding='utf-8')

document = loader.load()
print(document[0].metadata)
print(document[0].page_content)

{'source': 'D:\\RAG Projects\\Build-a-RAG-Chatbot\\data\\text_files\\python_intro.txt'}
Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.


#### Directory loader

In [20]:
from langchain_community.document_loaders import DirectoryLoader

In [28]:
dir_loder = DirectoryLoader(
    path=r'D:\RAG Projects\Build-a-RAG-Chatbot\data\text_files',
    glob= "**/*.txt",
    loader_cls= TextLoader,
    loader_kwargs= {'encoding' :'utf-8' },
    show_progress=True
)

document = dir_loder.load()

print(f"📁 Loaded {len(document)} documents")
for i, doc in enumerate(document):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 683.67it/s]

📁 Loaded 2 documents

Document 1:
  Source: D:\RAG Projects\Build-a-RAG-Chatbot\data\text_files\machine_learning.txt
  Length: 575 characters

Document 2:
  Source: D:\RAG Projects\Build-a-RAG-Chatbot\data\text_files\python_intro.txt
  Length: 489 characters


In [ ]:
text = document[0].page_content

char_splitter = CharacterTextSplitter(
    separator= '\n',
    chunk_size =200,
    chunk_overlap = 20,
    length_function = len
)


char_chunks=char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Created 4 chunks
First chunk: Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems...


In [39]:
print(char_chunks[0])
print("---------------")
print(char_chunks[1])
print("---------------")
print(char_chunks[2])
print("---------------")
print(char_chunks[3])

Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
---------------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
Types of Machine Learning:
---------------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
---------------
Applications include image recognition, speech processing, and recommendation systems


# RecursiveCharacterTextSplitter

In [42]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n","\n"," ",""],
    chunk_size = 200,
    chunk_overlap = 20,
    length_function = len
)

recursive_chunks=recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")

Created 6 chunks
First chunk: Machine Learning Basics...


In [44]:
print(recursive_chunks[0])
print("---------------")
print(recursive_chunks[1])
print("---------------")
print(recursive_chunks[2])
print("---------------")
print(recursive_chunks[4])
print("---------------")
print(recursive_chunks[5])

Machine Learning Basics
---------------
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
---------------
that can access data and use it to learn for themselves.
---------------
3. Reinforcement Learning: Learning through rewards and penalties
---------------
Applications include image recognition, speech processing, and recommendation systems


In [45]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")
    
    
    print()


Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'
Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'
Chunk 4: 'is sentence five. This is sentence six.'



In [46]:
# Method 3: Token-based splitting
print("\n3️⃣ TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=50,  # Size in tokens (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


3️⃣ TOKEN TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...
